In [1]:
# Librairies essentielles à importer
from pandas_datareader import data
import matplotlib.pyplot as plt
import pandas as pd
import datetime as dt
import urllib.request, json
import os
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
import missingno as msno
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from matplotlib.patches import Polygon

In [2]:
# Extraction des données depuis yahoo finance
!pip install yfinance pandas
import yfinance as yf

In [3]:
# Téléchargement des données journalières de la Société Générale sur plus de 25 ans
import yfinance as yf

ticker = "GLE.PA"

# Télécharger les données pour un seul ticker
df_GLE = yf.download(ticker, start="2000-01-01", end="2025-09-10", interval="1d")

/tmp/ipykernel_2000/420426704.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_GLE = yf.download(ticker, start="2000-01-01", end="2025-09-10", interval="1d")
[*********************100%***********************]  1 of 1 completed


In [4]:
df_GLE.columns = df_GLE.columns.get_level_values(0)


In [5]:
# Suppression des doublons
df_clean = df_GLE.drop_duplicates(keep ='first')

print(df_clean)

Price           Close       High        Low       Open   Volume
Date                                                           
2000-01-03  16.711756  17.341257  16.524406  17.303784  1078214
2000-01-04  16.636814  16.666789  16.344546  16.636814  2529529
2000-01-05  16.486931  16.636813  16.344545  16.337050  1233422
2000-01-06  16.337053  16.636817  16.299583  16.374524  1221423
2000-01-07  16.337053  16.561876  16.262113  16.262113   866415
...               ...        ...        ...        ...      ...
2025-09-03  51.497723  51.673217  50.776250  51.185734  2648836
2025-09-04  52.726185  52.940675  51.322231  51.322231  2331733
2025-09-05  51.712215  53.038170  51.497722  52.999171  2111572
2025-09-08  52.414192  52.433692  51.556224  51.926709  2302225
2025-09-09  52.960171  52.999170  51.790210  52.472688  2546754

[6590 rows x 5 columns]


In [6]:
train = pd.DataFrame(df_clean[0:int(len(df_clean)*0.70)])
test = pd.DataFrame(df_clean[int(len(df_clean)*0.70): int(len(df_clean))])

print(train.shape)
print(test.shape)

(4613, 5)
(1977, 5)


In [7]:
train.head()


Price,Close,High,Low,Open,Volume
Date,,,,,
2000-01-03,16.711756,17.341257,16.524406,17.303784,1078214
2000-01-04,16.636814,16.666789,16.344546,16.636814,2529529
2000-01-05,16.486931,16.636813,16.344545,16.337050,1233422
2000-01-06,16.337053,16.636817,16.299583,16.374524,1221423
2000-01-07,16.337053,16.561876,16.262113,16.262113,866415


In [8]:
test.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2017-12-19,29.418030,29.549300,29.152123,29.182415,3473876
2017-12-20,29.195875,29.441587,29.037677,29.434854,3440340
2017-12-21,29.485344,29.653639,28.933335,29.061239,3204366
2017-12-22,29.253098,29.606516,29.212706,29.303584,2052010
2017-12-27,29.128557,29.478612,29.020848,29.185777,1793578


In [9]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0,1))

In [10]:
train_close = train.iloc[:, 4:5].values
test_close = test.iloc[:, 4:5].values

In [11]:
data_training_array = scaler.fit_transform(train_close)
data_training_array

array([[0.03690524],
       [0.08658102],
       [0.04221772],
       ...,
       [0.14130901],
       [0.28408357],
       [0.10646044]])

In [12]:
x_train = []
y_train = []

for i in range(100, data_training_array.shape[0]):
    x_train.append(data_training_array[i-100: i])
    y_train.append(data_training_array[i, 0])

x_train, y_train = np.array(x_train), np.array(y_train)

In [13]:
x_train.shape


(4513, 100, 1)

In [14]:
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential

In [15]:
print(type(x_train.shape[1]), x_train.shape[1])
print(type(x_train.shape[2]), x_train.shape[2])


<class 'int'> 100
<class 'int'> 1


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

In [17]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [18]:
!pip install tensorflow==2.19.0 --force-reinstall


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 k

In [19]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

model = Sequential()
model.add(LSTM(units=50, activation='relu', return_sequences=True, input_shape=(100,1)))
model.add(Dropout(0.2))
model.add(LSTM(units=60, activation='relu', return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(units=80, activation='relu', return_sequences=True))
model.add(Dropout(0.4))
model.add(LSTM(units=120, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(units=1))

model.compile(loss='mean_squared_error', optimizer='adam')
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 50)        │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 50)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100, 60)        │        26,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 60)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100, 80)        │        45,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100, 80)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 120)            │        96,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           121 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 178,761 (698.29 KB)

 Trainable params: 178,761 (698.29 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Extract Close prices as NumPy array
data = df_clean[['Close']].values  # shape: (6590, 1)

# Train/test split (e.g., 80% for training)
split_index = int(len(data) * 0.8)  # ~5272 for training
train_data = data[:split_index]
test_data = data[split_index:]

# Define scaler and window size
scaler = MinMaxScaler()
smoothing_window_size = 2500

# Normalize training data in chunks
for i in range(0, len(train_data) - smoothing_window_size, smoothing_window_size):
    scaler.fit(train_data[i:i + smoothing_window_size])
    train_data[i:i + smoothing_window_size] = scaler.transform(train_data[i:i + smoothing_window_size])

# Normalize the remaining portion (last window)
scaler.fit(train_data[i + smoothing_window_size:])
train_data[i + smoothing_window_size:] = scaler.transform(train_data[i + smoothing_window_size:])

# Normalize test data with last trained scaler
test_data = scaler.transform(test_data)


In [21]:

class DataGeneratorSeq(object):

    def __init__(self,prices,batch_size,num_unroll):
        self._prices = prices
        self._prices_length = len(self._prices) - num_unroll
        self._batch_size = batch_size
        self._num_unroll = num_unroll
        self._segments = self._prices_length //self._batch_size
        self._cursor = [offset * self._segments for offset in range(self._batch_size)]

    def next_batch(self):

        batch_data = np.zeros((self._batch_size),dtype=np.float32)
        batch_labels = np.zeros((self._batch_size),dtype=np.float32)

        for b in range(self._batch_size):
            if self._cursor[b]+1>=self._prices_length:
                #self._cursor[b] = b * self._segments
                self._cursor[b] = np.random.randint(0,(b+1)*self._segments)

            batch_data[b] = self._prices[self._cursor[b]]
            batch_labels[b]= self._prices[self._cursor[b]+np.random.randint(0,5)]

            self._cursor[b] = (self._cursor[b]+1)%self._prices_length

        return batch_data,batch_labels

    def unroll_batches(self):

        unroll_data,unroll_labels = [],[]
        init_data, init_label = None,None
        for ui in range(self._num_unroll):

            data, labels = self.next_batch()

            unroll_data.append(data)
            unroll_labels.append(labels)

        return unroll_data, unroll_labels

    def reset_indices(self):
        for b in range(self._batch_size):
            self._cursor[b] = np.random.randint(0,min((b+1)*self._segments,self._prices_length-1))



dg = DataGeneratorSeq(train_data,5,5)
u_data, u_labels = dg.unroll_batches()

for ui,(dat,lbl) in enumerate(zip(u_data,u_labels)):
    print('\n\nUnrolled index %d'%ui)
    dat_ind = dat
    lbl_ind = lbl
    print('\tInputs: ',dat )
    print('\n\tOutput:',lbl)




Unrolled index 0
	Inputs:  [0.12125582 0.2969602  0.40317142 0.08777726 0.6020895 ]

	Output: [0.12000018 0.2960188  0.40317142 0.08777726 0.56703186]


Unrolled index 1
	Inputs:  [0.12000018 0.28817257 0.38564926 0.06313206 0.58100414]

	Output: [0.11497769 0.2960188  0.38564926 0.0802374  0.56703186]


Unrolled index 2
	Inputs:  [0.11748888 0.30323732 0.40341365 0.04715204 0.58265525]

	Output: [0.11497769 0.30323732 0.38177338 0.04715204 0.56982636]


Unrolled index 3
	Inputs:  [0.11497769 0.2988435  0.37676707 0.0802374  0.56703186]

	Output: [0.11497769 0.29633257 0.37676707 0.05930589 0.55001134]


Unrolled index 4
	Inputs:  [0.11497769 0.2960188  0.36610845 0.05930589 0.5608078 ]

	Output: [0.11497769 0.2988435  0.35867956 0.03871196 0.49247104]


/tmp/ipykernel_2000/3095774458.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  batch_data[b] = self._prices[self._cursor[b]]
/tmp/ipykernel_2000/3095774458.py:22: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  batch_labels[b]= self._prices[self._cursor[b]+np.random.randint(0,5)]


In [22]:
D = 1 # Dimensionality of the data. Since your data is 1-D this would be 1
num_unrollings = 50 # Number of time steps you look into the future.
batch_size = 500 # Number of samples in a batch
num_nodes = [200,200,150] # Number of hidden nodes in each layer of the deep LSTM stack we're using
n_layers = len(num_nodes) # number of layers
dropout = 0.2 # dropout amount

# tf.reset_default_graph() # This is important in case you run this multiple times

In [23]:
# Input data.
# In TensorFlow 2.x, placeholders are not used directly in this way.
# Input shape is defined in the first layer of the Keras model.
train_inputs, train_outputs = [],[]

# You unroll the input over time defining placeholders for each time step
# for ui in range(num_unrollings):
#     train_inputs.append(tf.placeholder(tf.float32, shape=[batch_size,D],name='train_inputs_%d'%ui))
#     train_outputs.append(tf.placeholder(tf.float32, shape=[batch_size,1], name = 'train_outputs_%d'%ui))

In [24]:
testing_array = scaler.transform(test)

x_test = []
y_test = []

for i in range(100, testing_array.shape[0]):
    x_test.append(testing_array[i-100:i])
    y_test.append(testing_array[i, 0])

x_test, y_test = np.array(x_test), np.array(y_test)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


ValueError: X has 5 features, but MinMaxScaler is expecting 1 features as input.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
import matplotlib.pyplot as plt

# --- 1. Chargement et split ---
# Assurez-vous que df_clean est déjà chargé avec une colonne 'Close'
train = pd.DataFrame(df_clean[0:int(len(df_clean)*0.70)])
test = pd.DataFrame(df_clean[int(len(df_clean)*0.70): int(len(df_clean))])

# --- 2. Normalisation sur la colonne 'Close' uniquement ---
scaler = MinMaxScaler(feature_range=(0,1))
train_close = train[['Close']]
test_close = test[['Close']]

training_array = scaler.fit_transform(train_close)
testing_array = scaler.transform(test_close)

# --- 3. Création des séquences ---
x_train, y_train = [], []

for i in range(100, len(training_array)):
    x_train.append(training_array[i-100:i])
    y_train.append(training_array[i])

x_train, y_train = np.array(x_train), np.array(y_train)

x_test, y_test = [], []

for i in range(100, len(testing_array)):
    x_test.append(testing_array[i-100:i])
    y_test.append(testing_array[i])

x_test, y_test = np.array(x_test), np.array(y_test)

# --- 4. Création du modèle LSTM ---
model = Sequential()
model.add(LSTM(units=50, activation='relu', return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(Dropout(0.2))

model.add(LSTM(units=60, activation='relu', return_sequences=True))
model.add(Dropout(0.3))

model.add(LSTM(units=80, activation='relu', return_sequences=True))
model.add(Dropout(0.4))

model.add(LSTM(units=120, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(units=1))

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# --- 5. Entraînement ---
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=50,
    batch_size=32
)

# --- 6. Courbe de perte (optionnel) ---
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# 1. Prédictions sur l'ensemble test
y_pred_scaled = model.predict(x_test)

# 2. Inversion du scaling (remise à l'échelle d'origine)
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1))

# 3. Calcul du MSE
mse = mean_squared_error(y_test_real, y_pred)

# 4. Visualisation des résultats
plt.figure(figsize=(14,6))
plt.plot(y_test_real, label='Prix réel (test)', color='blue')
plt.plot(y_pred, label='Prix prédit', color='red')
plt.title(f"Comparaison des prix réels vs prédits (MSE = {mse:.6f})", fontsize=14)
plt.xlabel("Temps")
plt.ylabel("Prix")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
